# 1. Read Libraries and Data

In [2]:
import pandas as pd
import numpy as np

import optuna
import tensorflow as tf

""" Sequential -  It allows you to create a model by simply adding layers one after another.
This is used for building simple neural networks where the flow of data is straightforward, 
from input to output, through a series of layers."""
from tensorflow.keras.models import Sequential

"""Dense is a fully connected layer, also known as a dense layer, in a neural network.
Each neuron in a dense layer receives input from all neurons in the previous layer."""
from tensorflow.keras.layers import Dense

"""Optimizers computes adaptive learning rates for each parameter"""
from tensorflow.keras.optimizers import SGD, Adam, RMSprop, Adagrad, Adadelta, Nadam

from sklearn.metrics import r2_score

data = pd.read_csv(r"C:/Users/aksta/Desktop/Salary_Data.csv")

data

C:\Users\aksta\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,Age,Gender,Education Level,Job Title,Years of Experience,Salary
0,32.0,Male,Bachelor's,Software Engineer,5.0,90000.0
1,28.0,Female,Master's,Data Analyst,3.0,65000.0
2,45.0,Male,PhD,Senior Manager,15.0,150000.0
3,36.0,Female,Bachelor's,Sales Associate,7.0,60000.0
4,52.0,Male,Master's,Director,20.0,200000.0
...,...,...,...,...,...,...
6699,49.0,Female,PhD,Director of Marketing,20.0,200000.0
6700,32.0,Male,High School,Sales Associate,3.0,50000.0
6701,30.0,Female,Bachelor's Degree,Financial Manager,4.0,55000.0
6702,46.0,Male,Master's Degree,Marketing Manager,14.0,140000.0


# 2. Descriptive Statistics

In [3]:
data.describe(include='all')

,Age,Gender,Education Level,Job Title,Years of Experience,Salary
count,6702.000000,6702,6701,6702,6701.000000,6699.000000
unique,NaN,3,7,193,NaN,NaN
top,NaN,Male,Bachelor's Degree,Software Engineer,NaN,NaN
freq,NaN,3674,2267,518,NaN,NaN
mean,33.620859,NaN,NaN,NaN,8.094687,115326.964771
std,7.614633,NaN,NaN,NaN,6.059003,52786.183911
min,21.000000,NaN,NaN,NaN,0.000000,350.000000
25%,28.000000,NaN,NaN,NaN,3.000000,70000.000000
50%,32.000000,NaN,NaN,NaN,7.000000,115000.000000
75%,38.000000,NaN,NaN,NaN,12.000000,160000.000000


# 3. Check missing values and drop them

In [4]:
data.isna().sum()

data.dropna(inplace=True)

# 4. Clean the Education Level column by merging category names into one using np.where .

In [5]:
data['Education Level'] = np.where(data['Education Level'] == "Bachelor's", "Bachelor's Degree", data['Education Level'])
data['Education Level'] = np.where(data['Education Level'] == "Master's", "Master's Degree", data['Education Level'])
data['Education Level'] = np.where(data['Education Level'] == "phD", "PhD", data['Education Level'])

data['Education Level'].value_counts()

Education Level
Bachelor's Degree    3021
Master's Degree      1860
PhD                  1369
High School           448
Name: count, dtype: int64

# 5. Create two new features from the Job Title column: 

- 1. the mean experience (exp_mean_by_job) for each job title
- 2. the number of unique education levels (edu_nunique_by_job) for each job title
- 3. then drop the Job Title column.

In [6]:
data['exp_mean_by_job'] = data.groupby('Job Title')['Years of Experience'].transform('mean')
data['edu_nunique_by_job'] = data.groupby('Job Title')['Education Level'].transform('nunique')

data.drop('Job Title', axis=1, inplace=True)

data

,Age,Gender,Education Level,Years of Experience,Salary,exp_mean_by_job,edu_nunique_by_job
0,32.0,Male,Bachelor's Degree,5.0,90000.0,4.449807,2
1,28.0,Female,Master's Degree,3.0,65000.0,4.969697,2
2,45.0,Male,PhD,15.0,150000.0,17.500000,1
3,36.0,Female,Bachelor's Degree,7.0,60000.0,1.428571,2
4,52.0,Male,Master's Degree,20.0,200000.0,20.000000,1
...,...,...,...,...,...,...,...
6699,49.0,Female,PhD,20.0,200000.0,12.068182,3
6700,32.0,Male,High School,3.0,50000.0,1.428571,2
6701,30.0,Female,Bachelor's Degree,4.0,55000.0,10.694030,4
6702,46.0,Male,Master's Degree,14.0,140000.0,9.454902,3


# 6. Declare target and inputs and Split data using train_test_split

In [7]:
from sklearn.model_selection import train_test_split

inputs = data.drop('Salary', axis=1)
target = data['Salary']

X_train, X_test, y_train, y_test = train_test_split(inputs, target, test_size=0.2, random_state=42)

# 7. Using LabelEncoder convert categorics to numerical

In [8]:
from sklearn.preprocessing import LabelEncoder

encoders = {}
for i in X_train.select_dtypes(include=['object', 'string']).columns:
    le = LabelEncoder()
    X_train[i] = le.fit_transform(X_train[i])
    encoders[i] = le

for i in X_test.columns:
    if i in encoders:
        le = encoders[i]
        X_test[i] = le.transform(X_test[i])

display(X_train)
display(X_test)

,Age,Gender,Education Level,Years of Experience,exp_mean_by_job,edu_nunique_by_job
3426,39.0,1,2,10.0,7.809524,2
1552,38.0,0,2,10.0,6.065574,3
6021,35.0,0,3,12.0,11.226667,1
4461,30.0,0,1,1.0,1.253521,1
3286,23.0,0,0,1.0,4.449807,2
...,...,...,...,...,...,...
3776,29.0,1,0,4.0,4.449807,2
5195,38.0,0,2,14.0,13.531250,1
5230,25.0,1,0,1.0,1.274510,1
5395,27.0,1,0,1.0,1.274510,1


,Age,Gender,Education Level,Years of Experience,exp_mean_by_job,edu_nunique_by_job
1883,36.0,1,2,9.0,6.065574,3
2630,27.0,0,2,4.0,4.149378,3
498,25.0,0,2,1.0,4.969697,2
5973,28.0,0,0,3.0,3.113636,1
4108,29.0,1,0,5.0,3.200000,2
...,...,...,...,...,...,...
2830,26.0,0,2,3.0,4.149378,3
6154,28.0,0,0,3.0,3.113636,1
4940,28.0,0,3,6.0,12.068182,3
135,39.0,0,0,10.0,9.000000,1


# 8. Scale inputs using StandartScaler

In [9]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

# Fit only on training data
scaler.fit(X_train)

# Transform train and test
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Convert back to DataFrames
X_train_scaled = pd.DataFrame(
    X_train_scaled,
    columns=X_train.columns,
    index=X_train.index
)

X_test_scaled = pd.DataFrame(
    X_test_scaled,
    columns=X_test.columns,
    index=X_test.index
)

display(X_train_scaled)
display(X_test_scaled)

,Age,Gender,Education Level,Years of Experience,exp_mean_by_job,edu_nunique_by_job
3426,0.707334,0.895071,0.624867,0.315738,-0.067456,-0.534691
1552,0.575653,-1.097364,0.624867,0.315738,-0.455648,0.493989
6021,0.180612,-1.097364,1.443458,0.648068,0.693177,-1.563370
4461,-0.477790,-1.097364,-0.193724,-1.179746,-1.526779,-1.563370
3286,-1.399553,-1.097364,-1.012315,-1.179746,-0.815307,-0.534691
...,...,...,...,...,...,...
3776,-0.609471,0.895071,-1.012315,-0.681251,-0.815307,-0.534691
5195,0.575653,-1.097364,0.624867,0.980398,1.206162,-1.563370
5230,-1.136192,0.895071,-1.012315,-1.179746,-1.522107,-1.563370
5395,-0.872832,0.895071,-1.012315,-1.179746,-1.522107,-1.563370


,Age,Gender,Education Level,Years of Experience,exp_mean_by_job,edu_nunique_by_job
1883,0.312293,0.895071,0.624867,0.149573,-0.455648,0.493989
2630,-0.872832,-1.097364,0.624867,-0.681251,-0.882181,0.493989
498,-1.136192,-1.097364,0.624867,-1.179746,-0.699583,-0.534691
5973,-0.741151,-1.097364,-1.012315,-0.847416,-1.112730,-1.563370
4108,-0.609471,0.895071,-1.012315,-0.515087,-1.093506,-0.534691
...,...,...,...,...,...,...
2830,-1.004512,-1.097364,0.624867,-0.847416,-0.882181,0.493989
6154,-0.741151,-1.097364,-1.012315,-0.847416,-1.112730,-1.563370
4940,-0.741151,-1.097364,1.443458,-0.348922,0.880493,0.493989
135,0.707334,-1.097364,-1.012315,0.315738,0.197536,-1.563370


# 9. Create ANN model for regression use with best hyperparameters using Optuna

- Last dense layer's activation function has to be relu
- loss='mae'
- metrics=['mae']
- metrics inside optuna and evaluate function has to be r2_score

In [10]:
def create_model(trial):
    # Building artificial neural network
    model = Sequential()

     # we add 2 hidden layers and 1 output layer
    model.add(Dense(units=trial.suggest_int('units_layer1', 16, 128), activation='relu'))
    model.add(Dense(units=trial.suggest_int('units_layer2', 16, 128), activation='relu'))
    model.add(Dense(units=1, activation='relu'))

    # Suggest hyperparameters for the optimizer
    optimizer_name = trial.suggest_categorical('optimizer', ['adam', 'rmsprop'])
    learning_rate = trial.suggest_float('learning_rate', 1e-2, 1.0, log=True)
    
    if optimizer_name == 'adam':
        optimizer = Adam(learning_rate=learning_rate)
    elif optimizer_name == 'rmsprop':
        optimizer = RMSprop(learning_rate=learning_rate)

    model.compile(optimizer=optimizer, loss='mae', metrics=['mae'])
    
    return model

def optimal(trial):
    
    # Suggest the number of epochs and batch size
    epochs = trial.suggest_int('epochs', 50, 200)
    batch_size = trial.suggest_int('batch_size', 16, 64)
    
    model = create_model(trial)
    
    history = model.fit(X_train_scaled, y_train, epochs=epochs, batch_size=batch_size)
    
    y_pred = model.predict(X_test_scaled)
    score = r2_score(y_test, y_pred)
    
    return score

study = optuna.create_study(direction='maximize')
study.optimize(optimal, n_trials=30)

print(f"Best trial: {study.best_trial.value}")
print(f"Best hyperparameters: {study.best_trial.params}")

best_params = study.best_trial.params

best_params

[I 2026-10-20 12:52:05,120] A new study created in memory with name: no-name-9fb87f78-8b38-4039-b497-8c4f770e973f


Epoch 1/128
84/84 ━━━━━━━━━━━━━━━━━━━━ 4s 12ms/step - loss: 76406.8281 - mae: 76406.8281
Epoch 2/128
84/84 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 24854.6914 - mae: 24854.6914
Epoch 3/128
84/84 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 21298.6973 - mae: 21298.6973
Epoch 4/128
84/84 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 20847.5586 - mae: 20847.5586
Epoch 5/128
84/84 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 20610.7598 - mae: 20610.7598
Epoch 6/128
84/84 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 20441.1562 - mae: 20441.1562
Epoch 7/128
84/84 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 20327.0273 - mae: 20327.0273
Epoch 8/128
84/84 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 20221.9355 - mae: 20221.9355
Epoch 9/128
84/84 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 20121.0742 - mae: 20121.0742
Epoch 10/128
84/84 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 20039.2109 - mae: 20039.2109
Epoch 11/128
84/84 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 19980.8711 - mae: 19980.8711
Epoch 12/128
84/84 ━━━━━━━━━━

[I 2026-10-20 12:52:46,123] Trial 0 finished with value: 0.8757333911043061 and parameters: {'epochs': 128, 'batch_size': 64, 'units_layer1': 118, 'units_layer2': 98, 'optimizer': 'adam', 'learning_rate': 0.029103708184859606}. Best is trial 0 with value: 0.8757333911043061.


Epoch 1/179
215/215 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - loss: 28366.7852 - mae: 28366.7852
Epoch 2/179
215/215 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 21628.4043 - mae: 21628.4043
Epoch 3/179
215/215 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 20021.3730 - mae: 20021.3730
Epoch 4/179
215/215 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 19234.8086 - mae: 19234.8086
Epoch 5/179
215/215 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 18825.0684 - mae: 18825.0703
Epoch 6/179
215/215 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 18465.9551 - mae: 18465.9570
Epoch 7/179
215/215 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 17252.5605 - mae: 17252.5605
Epoch 8/179
215/215 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 16145.3809 - mae: 16145.3809
Epoch 9/179
215/215 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 15233.4688 - mae: 15233.4688
Epoch 10/179
215/215 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 15125.2500 - mae: 15125.2500
Epoch 11/179
215/215 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 14212.5391 - mae: 14212.5391
Epoch 12

[I 2026-10-20 12:54:21,393] Trial 1 finished with value: 0.927542719015739 and parameters: {'epochs': 179, 'batch_size': 25, 'units_layer1': 107, 'units_layer2': 66, 'optimizer': 'adam', 'learning_rate': 0.5219038590537444}. Best is trial 1 with value: 0.927542719015739.


...

[I 2026-10-20 13:15:35,214] Trial 28 finished with value: 0.9126069452187145 and parameters: {'epochs': 181, 'batch_size': 16, 'units_layer1': 102, 'units_layer2': 90, 'optimizer': 'adam', 'learning_rate': 0.3486662896693304}. Best is trial 13 with value: 0.9435962770533816.


Epoch 1/159
256/256 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 28848.9316 - mae: 28848.9316
Epoch 2/159
256/256 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 21033.5684 - mae: 21033.5684
Epoch 3/159
256/256 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 20654.8770 - mae: 20654.8770
Epoch 4/159
256/256 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 20471.2090 - mae: 20471.2090
Epoch 5/159
256/256 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 20198.3730 - mae: 20198.3730
Epoch 6/159
256/256 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 20181.3730 - mae: 20181.3730
Epoch 7/159
256/256 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 20157.8477 - mae: 20157.8457
Epoch 8/159
256/256 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 20211.1914 - mae: 20211.1914
Epoch 9/159
256/256 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 20091.3965 - mae: 20091.3965
Epoch 10/159
256/256 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 19388.7344 - mae: 19388.7324
Epoch 11/159
256/256 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 17585.7129 - mae: 17585.7129
Epoch 12

[I 2026-10-20 13:16:50,623] Trial 29 finished with value: 0.9380337474346381 and parameters: {'epochs': 159, 'batch_size': 21, 'units_layer1': 116, 'units_layer2': 97, 'optimizer': 'adam', 'learning_rate': 0.14187455163534665}. Best is trial 13 with value: 0.9435962770533816.


Best trial: 0.9435962770533816
Best hyperparameters: {'epochs': 166, 'batch_size': 39, 'units_layer1': 87, 'units_layer2': 78, 'optimizer': 'adam', 'learning_rate': 0.3085186122202393}


{'epochs': 166,
 'batch_size': 39,
 'units_layer1': 87,
 'units_layer2': 78,
 'optimizer': 'adam',
 'learning_rate': 0.3085186122202393}

# 10. Train and evaluate the final model with the best hyperparameters

In [11]:
# Train the final model with the best hyperparameters

best_model = Sequential()
best_model.add(Dense(units=best_params['units_layer1'], activation='relu'))
best_model.add(Dense(units=best_params['units_layer2'], activation='relu'))
best_model.add(Dense(1, activation='relu'))

if best_params['optimizer'] == 'adam':
    best_optimizer = Adam(learning_rate=best_params['learning_rate'])
elif best_params['optimizer'] == 'sgd':
    best_optimizer = SGD(learning_rate=best_params['learning_rate'])
elif best_params['optimizer'] == 'rmsprop':
    best_optimizer = RMSprop(learning_rate=best_params['learning_rate'])
elif best_params['optimizer'] == 'adagrad':
    best_optimizer = Adagrad(learning_rate=best_params['learning_rate'])

best_model.compile(optimizer=best_optimizer, loss='mae', metrics=['mae'])


def evaluate(model, X_train, y_train, X_test, y_test):
    
    model.fit(X_train, y_train, epochs=best_params['epochs'], batch_size=best_params['batch_size'])
    
    '''Predictions for the training set'''
    
    y_train_pred = model.predict(X_train)

    '''Predictions for the test set'''
    
    y_test_pred = model.predict(X_test)

    '''Calculate metrics for the training set''' 
    
    r2_train = r2_score(y_train, y_train_pred)

    '''Calculate metrics for the test set'''
    
    r2_test = r2_score(y_test, y_test_pred)
    

    results = pd.DataFrame({
        'Dataset': ['Train', 'Test'],
        'R2': [r2_train, r2_test],
    
    })

    return results


evaluate(best_model, X_train_scaled, y_train, X_test_scaled, y_test)

Epoch 1/166
138/138 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 29921.7441 - mae: 29921.7441
Epoch 2/166
138/138 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 21926.0156 - mae: 21926.0176
Epoch 3/166
138/138 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 21123.0273 - mae: 21123.0254
Epoch 4/166
138/138 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 21066.0059 - mae: 21066.0059
Epoch 5/166
138/138 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 20809.1230 - mae: 20809.1211
Epoch 6/166
138/138 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 19847.8242 - mae: 19847.8242
Epoch 7/166
138/138 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 18707.4512 - mae: 18707.4512
Epoch 8/166
138/138 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 19097.4434 - mae: 19097.4414
Epoch 9/166
138/138 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 17431.1309 - mae: 17431.1309
Epoch 10/166
138/138 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 16399.7637 - mae: 16399.7637
Epoch 11/166
138/138 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 16449.0039 - mae: 16449.0039
Epoch 12

,Dataset,R2
0,Train,0.944383
1,Test,0.930239
